In [1]:
import pandas as pd
import numpy as np
import torch
import time
import copy
import math
from datetime import datetime
from torch.utils.data import DataLoader

In [2]:
class Feedforward(torch.nn.Module):
        def __init__(self, input_size, hidden_size, dropout):
            super(Feedforward, self).__init__()
            self.input_size = input_size
            self.hidden_size  = hidden_size
            self.fc1 = torch.nn.Linear(self.input_size, self.hidden_size)
            self.fc2 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc3 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc4 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc5 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            #self.relu = torch.nn.ReLU()
            self.relu = torch.nn.LeakyReLU(0.1)
            self.fc_out = torch.nn.Linear(self.hidden_size, 1)
            self.sigmoid = torch.nn.Sigmoid()
            self.dropout = torch.nn.Dropout(dropout)
            
        def forward(self, x):
            x = self.fc1(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc2(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc3(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc4(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc5(x)
            x = self.dropout(x)
            x = self.relu(x)
            output = self.fc_out(x)
            output = self.sigmoid(output)
            return output

In [3]:
data = pd.read_csv('/Users/nkerstingadxnet.com/Documents/Higgs/orig/atlas-higgs-challenge-2014-v2.csv')
data['Binary_Label'] = data['Label'].map({'s':1,'b':0})

In [4]:
nonphi_columns = [c for c in data.columns if "phi" in c]
for col in nonphi_columns:
    data.drop(col, axis=1, inplace=True)
data.head()

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_all_pt,Weight,Label,KaggleSet,KaggleWeight,Binary_Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,67.435,2.150,46.062,1.24,113.497,0.000814,s,t,0.002653,1
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,46.226,0.725,-999.000,-999.00,46.226,0.681042,b,t,2.233584,0
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,44.251,2.053,-999.000,-999.00,44.251,0.715742,b,t,2.347389,0
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,-999.000,-999.000,-999.000,-999.00,-0.000,1.660654,b,t,5.446378,0
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,-999.000,-999.000,-999.000,-999.00,0.000,1.904263,b,t,6.245333,0


In [5]:
# now replace all the '-999' values with NULL so we can properly handle them
#nulled_data = data.replace(-999.0, np.nan)
nulled_data = data.replace(-999.0, 0)
train_data = nulled_data.loc[nulled_data['KaggleSet'] == 't'].copy(deep=True)
public_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'b'].copy(deep=True)
private_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'v'].copy(deep=True)

In [6]:
orig_train_data = train_data.copy(deep=True)
orig_public_test_data = public_test_data.copy(deep=True)
orig_private_test_data = private_test_data.copy(deep=True)


# now replace the null values with column averages
train_data.drop('EventId', axis=1, inplace=True)
train_data.drop('Label', axis=1, inplace=True)
train_data.drop('Binary_Label', axis=1, inplace=True)
train_data.drop('Weight', axis=1, inplace=True)
train_data.drop('KaggleWeight', axis=1, inplace=True)
train_data.drop('KaggleSet', axis=1, inplace=True)
for col in train_data:
    train_data[col].fillna(train_data[col].mean(), inplace=True)
#train_data.isnull().sum()


public_test_data.drop('EventId', axis=1, inplace=True)
public_test_data.drop('Label', axis=1, inplace=True)
public_test_data.drop('Binary_Label', axis=1, inplace=True)
public_test_data.drop('Weight', axis=1, inplace=True)
public_test_data.drop('KaggleWeight', axis=1, inplace=True)
public_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in public_test_data:
    public_test_data[col].fillna(public_test_data[col].mean(), inplace=True)
#public_test_data.isnull().sum()


private_test_data.drop('EventId', axis=1, inplace=True)
private_test_data.drop('Label', axis=1, inplace=True)
private_test_data.drop('Binary_Label', axis=1, inplace=True)
private_test_data.drop('Weight', axis=1, inplace=True)
private_test_data.drop('KaggleWeight', axis=1, inplace=True)
private_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in private_test_data:
    private_test_data[col].fillna(private_test_data[col].mean(), inplace=True)
#private_test_data.isnull().sum()

In [7]:
# now let's normalize
normed_train_data = (train_data - train_data.min())/(train_data.max() - train_data.min())

normed_public_test_data = (public_test_data - public_test_data.min())/(public_test_data.max() - public_test_data.min())

normed_private_test_data = (private_test_data - private_test_data.min())/(private_test_data.max() - private_test_data.min())



In [8]:
train_input_data = []
for i in range(len(orig_train_data)):
   train_input_data.append([torch.tensor(normed_train_data.iloc[i], dtype=torch.float), torch.tensor(orig_train_data['Binary_Label'].iloc[i] , dtype=torch.float)])
valid_input_data = []
for i in range(len(orig_public_test_data)):
   valid_input_data.append([torch.tensor(normed_public_test_data.iloc[i], dtype=torch.float), torch.tensor(orig_public_test_data['Binary_Label'].iloc[i] , dtype=torch.float)])

In [9]:
train_dataloader = DataLoader(train_input_data, batch_size=128, shuffle=True)
valid_dataloader = DataLoader(valid_input_data, batch_size=128, shuffle=True)

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = 'cpu'
model = Feedforward(24, 600, 0.05)
model.to(device)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

In [12]:
now = datetime.now()
dt_string = now.strftime("%d%m%Y%H%M%S")
logfile = open(dt_string + ".log",'w')
logfile.write(f"Start of Log\n")
logfile.write(f"Now training with device={device}\n")
#logfile.write(f"Num Devices: {torch.cuda.device_count()}, DEVICE NAME 0: {torch.cuda.get_device_name(0)}\n")

start_time = time.time()
epoch = 1
maxvalcount = 5
minval_loss = np.inf
valcount = maxvalcount
best_model = model
logfile.write(f"Using model with params on CUDA: {next(model.parameters()).is_cuda}")
for epoch in range(epoch):
    outstring = f"EPOCH: {epoch}"
    print(outstring)
    logfile.write(outstring + '\n')
    model.train()
    for i,batch in enumerate(train_dataloader):
        #logfile.write(f"Now at i={i}\n")
        inputs, output = batch
        inputs, output = inputs.to(device), output.to(device)
        #logfile.write(f"Data on device\n")
        optimizer.zero_grad()
        logfile.write(f"Optimizer zeroed\n")
        # Forward pass
        y_pred = model(inputs)
        logfile.write(f"Prediction done\n")
        # Compute Loss
        loss = criterion(y_pred.squeeze(), output)
        #logfile.write(f"Loss done\n")
        if i % 100 == 0:
            logstring = 'Batch {}: train loss: {}'.format(i, loss.item())
            print(logstring)
            logfile.write(logstring + '\n')
        # Backward pass
        loss.backward()
        #logfile.write(f"Loss backpropped\n")
        optimizer.step()
    # compute validation loss
    model.eval()
    with torch.set_grad_enabled(False):
        val_loss = 0
        for i,batch in enumerate(valid_dataloader):
            inputs, output = batch
            inputs, output = inputs.to(device), output.to(device)
            y_pred = model(inputs)
            loss = criterion(y_pred.squeeze(), output)
            val_loss += loss
            if i % 100 == 0:
                logstring = 'Validation Batch {}: loss: {}'.format(i, loss.item())
                print(logstring)
                logfile.write(logstring + '\n')
        avg_val_loss = val_loss / len(valid_dataloader)
        valstring = f"AVERAGE BATCH VAL LOSS = {avg_val_loss}"
        print(valstring)
        logfile.write(valstring + '\n')
    if avg_val_loss < minval_loss:
        minval_loss = avg_val_loss
        best_model = copy.deepcopy(model)
        valcount = maxvalcount
    else:
        valcount -= 1
    if valcount == 0:
        endmsg = f"Validation Loss failed to decrease in {maxvalcount} epochs, exiting with best model, val loss = {minval_loss}"
        print(endmsg)
        logfile.write(endmsg + '\n')
        break

end_time = time.time()
timestring = f"Time elapsed in training: {end_time - start_time} seconds"
print(timestring)
logfile.write(timestring + '\n')

EPOCH: 0
Batch 0: train loss: 0.6940249800682068
Batch 100: train loss: 0.6629412174224854
Batch 200: train loss: 0.6291763782501221
Batch 300: train loss: 0.6656692028045654
Batch 400: train loss: 0.6453731060028076
Batch 500: train loss: 0.6444346904754639
Batch 600: train loss: 0.6434956789016724
Batch 700: train loss: 0.6158220767974854
Batch 800: train loss: 0.5946048498153687
Batch 900: train loss: 0.657804012298584
Batch 1000: train loss: 0.607974112033844
Batch 1100: train loss: 0.6677079200744629
Batch 1200: train loss: 0.6526495218276978
Batch 1300: train loss: 0.6165100336074829
Batch 1400: train loss: 0.6483492255210876
Batch 1500: train loss: 0.6683709025382996
Batch 1600: train loss: 0.6430993676185608
Batch 1700: train loss: 0.6785418391227722
Batch 1800: train loss: 0.6946463584899902
Batch 1900: train loss: 0.5870494246482849
Validation Batch 0: loss: 0.7119989991188049
Validation Batch 100: loss: 0.6368816494941711
Validation Batch 200: loss: 0.6328816413879395
Valida

53